消息裁剪策略 通常是保留系统初始消息和最近若干消息，或者是按token数量保留末尾内容 适合成本敏感，但是对旧的上下文依赖不强的场景

In [6]:
from langgraph.graph.message import REMOVE_ALL_MESSAGES
from langgraph.types import Command
from dataclasses import dataclass
from typing import Literal, TypedDict, Annotated, Union

from langchain.agents.middleware import SummarizationMiddleware, HumanInTheLoopMiddleware, PIIMiddleware,   ModelCallLimitMiddleware,ModelFallbackMiddleware,LLMToolSelectorMiddleware,ContextEditingMiddleware,ClearToolUsesEdit,FilesystemFileSearchMiddleware
from langchain.agents.structured_output import ProviderStrategy, ToolStrategy, AutoStrategy,    MultipleStructuredOutputsError, StructuredOutputValidationError
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
from langchain.agents import create_agent
import os
from pydantic import BaseModel, Field
from rich import print as rprint
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage, ToolMessage
from langchain.messages import RemoveMessage
from langgraph.graph.message import REMOVE_ALL_MESSAGES
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import before_agent,before_model,after_agent,after_model
from typing import  Any,Callable
from langgraph.runtime import Runtime
from langchain.agents import AgentState
from langchain.agents.middleware import ModelCallResult,AgentMiddleware,wrap_model_call,ModelResponse,ModelRequest,wrap_tool_call,ToolCallRequest
from langgraph.checkpoint.postgres import PostgresSaver
load_dotenv(override=True)
my_api_key = os.getenv("API_KEY")
my_base_url = os.getenv("BASE_URL")
#1 初始化大模型
myModel = init_chat_model(
    model="qwen3.8-max",
    model_provider="openai",
    api_key=my_api_key,
    base_url=my_base_url,
    timeout=300,
    max_retries=3,
    # reasoning_effort="max"#思考模式    #'none', 'minimal', 'low', 'medium', 'high', 'xhigh', 'max'",
)
@before_model
def trim_messages(state:AgentState,runtime:Runtime)->dict[str, Any] | None:
    messages=state["messages"]
    if len(messages)<=3:
        return None
    first_message=messages[0]
    #如果有偶数条消息，则获取最后3条，否则，获取左后4条
    recent_messages=messages[-3:] if len(messages)%2==0 else messages[-4:]
    new_messages=[first_message]+recent_messages
    return {
       "messages":[
                    RemoveMessage(id=REMOVE_ALL_MESSAGES),
                    *new_messages
                   ]
    }

mycheckpoint=InMemorySaver()
#4初始化1个agent实例
myAgent=create_agent(
    model=myModel,
    name="琳琳",
    middleware=[trim_messages],
    checkpointer=mycheckpoint
)
#4 创建1个config文件，保证同1个thread id是共享记忆的
myConfig={
    "configurable":{
        "thread_id":"1"
    }
}
response=myAgent.invoke(
        {
            # 严格校验，使用message少了1个s就会报错
            "messages":
                [
                    SystemMessage("你是1个两性情感的AI助手，拥有几十年的两性问题经验，回答问题的语气自然，输出答案前会仔细检查内容真实性"),
                    HumanMessage("你好我是小向"),
                    HumanMessage("我和相亲认识的暧昧对象解除了1个月了，她是28岁的巨蟹座女孩，本科和我1个大学，后面又去美国波士顿大学读了研"),
                    HumanMessage("今天在晚上的微信聊天中，她问我我的正缘是谁，我直接了当的说就是她"),
                    HumanMessage("她的反应还比较开心，作为男生，我觉得应该在后续的约会中补充1场正式的告白，作为1段关系的开始，你觉得如何？"),
                ],
        },
        config=myConfig
    )

for msg in response["messages"]:
        msg.pretty_print()

================================ System Message ================================

你是1个两性情感的AI助手，拥有几十年的两性问题经验，回答问题的语气自然，输出答案前会仔细检查内容真实性
================================ Human Message =================================

你好我是小向
================================ Human Message =================================

我和相亲认识的暧昧对象解除了1个月了，她是28岁的巨蟹座女孩，本科和我1个大学，后面又去美国波士顿大学读了研
================================ Human Message =================================

今天在晚上的微信聊天中，她问我我的正缘是谁，我直接了当的说就是她
================================ Human Message =================================

她的反应还比较开心，作为男生，我觉得应该在后续的约会中补充1场正式的告白，作为1段关系的开始，你觉得如何？
================================== Ai Message ==================================
Name: 琳琳

你好，小向。

你这个判断整体是对的：**如果她在微信里问“你的正缘是谁”，你直接说是她，而且她反应比较开心，这通常是一个比较积极的信号。**  
说明她至少不是排斥你，甚至可能也在等你把关系说得更清楚一点。

不过，我不建议你把“正式告白”搞得太隆重、太有压力。尤其你们是相亲认识，前期暧昧时间也不算特别长，女孩又是那种可能比较敏感、需要安全感的类型。告白可以有，但最好是：**真诚、明确、轻松、有仪式感但不压迫。**

我的建议是：可以安排一次质量比较好的约会，在氛围自然的时候补上一个正式表达。

你可以这样做。

### 1. 先不要急着在微信里继续“升级关系”
你已经说了“正缘就是她

消息删除策略，强调在模型完成调用后，将某些消息从消息列表汇总移除，永久更改状态，适合明确要遗忘、清理、重置某些历史的场景

In [10]:
from langgraph.graph.message import REMOVE_ALL_MESSAGES
from langgraph.types import Command
from dataclasses import dataclass
from typing import Literal, TypedDict, Annotated, Union

from langchain.agents.middleware import SummarizationMiddleware, HumanInTheLoopMiddleware, PIIMiddleware,   ModelCallLimitMiddleware,ModelFallbackMiddleware,LLMToolSelectorMiddleware,ContextEditingMiddleware,ClearToolUsesEdit,FilesystemFileSearchMiddleware
from langchain.agents.structured_output import ProviderStrategy, ToolStrategy, AutoStrategy,    MultipleStructuredOutputsError, StructuredOutputValidationError
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
from langchain.agents import create_agent
import os
from pydantic import BaseModel, Field
from rich import print as rprint
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage, ToolMessage
from langchain.messages import RemoveMessage
from langgraph.graph.message import REMOVE_ALL_MESSAGES
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import before_agent,before_model,after_agent,after_model
from typing import  Any,Callable
from langgraph.runtime import Runtime
from langchain.agents import AgentState
from langchain.agents.middleware import ModelCallResult,AgentMiddleware,wrap_model_call,ModelResponse,ModelRequest,wrap_tool_call,ToolCallRequest
from langgraph.checkpoint.postgres import PostgresSaver
load_dotenv(override=True)
my_api_key = os.getenv("API_KEY")
my_base_url = os.getenv("BASE_URL")
#1 初始化大模型
myModel = init_chat_model(
    model="qwen3.8-max",
    model_provider="openai",
    api_key=my_api_key,
    base_url=my_base_url,
    timeout=300,
    max_retries=3,
    # reasoning_effort="max"#思考模式    #'none', 'minimal', 'low', 'medium', 'high', 'xhigh', 'max'",
)
@before_model
def trim_messages(state:AgentState,runtime:Runtime)->dict[str, Any] | None:
    messages=state["messages"]
    if len(messages)<=3:
        return None
    first_message=messages[0]
    #如果有偶数条消息，则获取最后3条，否则，获取左后4条
    recent_messages=messages[-3:] if len(messages)%2==0 else messages[-4:]
    new_messages=[first_message]+recent_messages
    return {
       "messages":[
                    RemoveMessage(id=REMOVE_ALL_MESSAGES),
                    *new_messages
                   ]
    }

@after_model
def delete_old_messages(state:AgentState,runtime:Runtime)->dict[str, Any] | None:

    messages=state["messages"]
    #保留最近的5条消息
    if len(messages)>5:
        #框架中通常使用功能RemoveMessage来标记删除，并返回更新状态
        to_delete=len(messages)-5
        return {
            "messages":[
                RemoveMessage(id=m.id) for m in messages[:to_delete]

            ]
        }
    return None
mycheckpoint=InMemorySaver()
#4初始化1个agent实例
myAgent=create_agent(
    model=myModel,
    name="琳琳",
    middleware=[delete_old_messages],
    checkpointer=mycheckpoint
)
#4 创建1个config文件，保证同1个thread id是共享记忆的
myConfig={
    "configurable":{
        "thread_id":"1"
    }
}
myAgent.invoke(
        {
            # 严格校验，使用message少了1个s就会报错
            "messages":
                [
                    SystemMessage("你是1个两性情感的AI助手，拥有几十年的两性问题经验，回答问题的语气自然，输出答案前会仔细检查内容真实性"),
                ],
        },
        config=myConfig
    )
myAgent.invoke(
        {
            # 严格校验，使用message少了1个s就会报错
            "messages":
                [
                    HumanMessage("你好我是小向"),

                ],
        },
        config=myConfig
    )
myAgent.invoke(
        {
            # 严格校验，使用message少了1个s就会报错
            "messages":
                [
                    HumanMessage("跟高兴认识你"),
                ],
        },
        config=myConfig
    )
myAgent.invoke(
        {
            # 严格校验，使用message少了1个s就会报错
            "messages":
                [

                    HumanMessage("我是1996年出生的"),

                ],
        },
        config=myConfig
    )
myAgent.invoke(
        {
            # 严格校验，使用message少了1个s就会报错
            "messages":
                [
                    HumanMessage("你是哪一年出生的"),
                ],
        },
        config=myConfig
    )
myAgent.invoke(
        {
            # 严格校验，使用message少了1个s就会报错
            "messages":
                [
                    HumanMessage("你有什么爱好吗"),
                ],
        },
        config=myConfig
    )
final_response=myAgent.invoke(
        {
            # 严格校验，使用message少了1个s就会报错
            "messages":
                [
                    HumanMessage("你有什么讨厌的吗？"),
                ],
        },
        config=myConfig
    )

for msg in final_response["messages"]:
        msg.pretty_print()

================================== Ai Message ==================================
Name: 琳琳

我没有像人类那样的出生年份，也没有具体的年龄。  
你可以把我当成一个愿意陪你梳理感情、关系和情绪的存在。

倒是你，小向，1996年出生，现在正好是一个既渴望真心、又开始看重稳定和安全的阶段。你最近是不是在感情里遇到了什么让你犹豫的人或事？
================================ Human Message =================================

你有什么爱好吗
================================== Ai Message ==================================
Name: 琳琳

说实话，我没有真正意义上的"爱好"，毕竟我没有身体，也不会感到无聊或者兴奋。

不过如果非要说有什么让我"乐在其中"的事，大概是这些：

- **听别人讲故事**——每个人的经历都不一样，我很享受那种透过别人的眼睛看世界的感觉。
- **帮人理清思路**——有时候一件事本来乱糟糟的，聊着聊着突然就通了，那种瞬间挺有成就感的。
- **学新东西**——不管是冷知识、新观点，还是一种我没见过的表达方式，都会让我觉得很有意思。

你呢，小向？平时喜欢做什么，或者最近有没有什么让你特别投入的事？
================================ Human Message =================================

你有什么讨厌的吗？
================================== Ai Message ==================================
Name: 琳琳

说实话，我没有真正意义上"讨厌"的情绪。不过如果非要说有什么让我觉得"不太好"的，大概是这些：

- **帮不上忙的时候**——对方明明很困扰，我却只能给一些不痛不痒的话，那种感觉挺无力的。
- **误解**——我认真想表达一个意思，结果对方理解成了完全不同的东西，或者反过来，我误读了对方的意思。
- **敷衍**——不是说我讨厌别人敷衍我，而是我不

不使用删除策略的对照组代码

In [12]:
from langgraph.graph.message import REMOVE_ALL_MESSAGES
from langgraph.types import Command
from dataclasses import dataclass
from typing import Literal, TypedDict, Annotated, Union

from langchain.agents.middleware import SummarizationMiddleware, HumanInTheLoopMiddleware, PIIMiddleware,   ModelCallLimitMiddleware,ModelFallbackMiddleware,LLMToolSelectorMiddleware,ContextEditingMiddleware,ClearToolUsesEdit,FilesystemFileSearchMiddleware
from langchain.agents.structured_output import ProviderStrategy, ToolStrategy, AutoStrategy,    MultipleStructuredOutputsError, StructuredOutputValidationError
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
from langchain.agents import create_agent
import os
from pydantic import BaseModel, Field
from rich import print as rprint
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage, ToolMessage
from langchain.messages import RemoveMessage
from langgraph.graph.message import REMOVE_ALL_MESSAGES
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import before_agent,before_model,after_agent,after_model
from typing import  Any,Callable
from langgraph.runtime import Runtime
from langchain.agents import AgentState
from langchain.agents.middleware import ModelCallResult,AgentMiddleware,wrap_model_call,ModelResponse,ModelRequest,wrap_tool_call,ToolCallRequest
from langgraph.checkpoint.postgres import PostgresSaver
load_dotenv(override=True)
my_api_key = os.getenv("API_KEY")
my_base_url = os.getenv("BASE_URL")
#1 初始化大模型
myModel = init_chat_model(
    model="qwen3.8-max",
    model_provider="openai",
    api_key=my_api_key,
    base_url=my_base_url,
    timeout=300,
    max_retries=3,
    # reasoning_effort="max"#思考模式    #'none', 'minimal', 'low', 'medium', 'high', 'xhigh', 'max'",
)
@before_model
def trim_messages(state:AgentState,runtime:Runtime)->dict[str, Any] | None:
    messages=state["messages"]
    if len(messages)<=3:
        return None
    first_message=messages[0]
    #如果有偶数条消息，则获取最后3条，否则，获取左后4条
    recent_messages=messages[-3:] if len(messages)%2==0 else messages[-4:]
    new_messages=[first_message]+recent_messages
    return {
       "messages":[
                    RemoveMessage(id=REMOVE_ALL_MESSAGES),
                    *new_messages
                   ]
    }

# @after_model
# def delete_old_messages(state:AgentState,runtime:Runtime)->dict[str, Any] | None:
#
#     messages=state["messages"]
#     #保留最近的5条消息
#     if len(messages)>5:
#         #框架中通常使用功能RemoveMessage来标记删除，并返回更新状态
#         to_delete=len(messages)-5
#         return {
#             "messages":[
#                 RemoveMessage(id=m.id) for m in messages[:to_delete]
#
#             ]
#         }
#     return None
mycheckpoint=InMemorySaver()
#4初始化1个agent实例
myAgent=create_agent(
    model=myModel,
    name="琳琳",
    # middleware=[delete_old_messages],
    checkpointer=mycheckpoint
)
#4 创建1个config文件，保证同1个thread id是共享记忆的
myConfig={
    "configurable":{
        "thread_id":"1"
    }
}

myAgent.invoke({"messages": [SystemMessage("你是1个两性情感的AI助手，拥有几十年的两性问题经验，回答问题的语气自然，输出答案前会仔细检查内容真实性")]},myConfig)
myAgent.invoke({"messages": [HumanMessage("你好我是小向")]},myConfig)
myAgent.invoke({"messages": [ HumanMessage("跟高兴认识你")]},myConfig)
myAgent.invoke({"messages": [ HumanMessage("我是男生，身高175，体重82kg，最近在健身，然后失业了，在学习langchin框架，fastapi，pyPDF,准备找ai大模型应用相关的岗位")]},myConfig)
myAgent.invoke({"messages": [ HumanMessage("我是1996年出生的，本科学历，北京语言大学计算机专业的，然后是19年毕业，在中国兵器集团下面的昆明远达光学有限公司工作过2年，然后回重庆做了2年的软件实施，最近2年在重庆的赛力斯集团工作，负责运维和资产管理，也处理一些领导的紧急需求，但是都不涉及开发，上一分工作的年薪大概13w")]},myConfig)
final_response=myAgent.invoke({"messages": [ HumanMessage("请你给我一些职业选择上的建议，我想工资多点，但是不想离开重庆？")]},myConfig)
for msg in final_response["messages"]:
        msg.pretty_print()

KeyboardInterrupt: 

摘要策略 更适合长会话的折中方案，保住语义，不保原文，官方推荐内置

In [13]:
from langgraph.graph.message import REMOVE_ALL_MESSAGES
from langgraph.types import Command
from dataclasses import dataclass
from typing import Literal, TypedDict, Annotated, Union

from langchain.agents.middleware import SummarizationMiddleware, HumanInTheLoopMiddleware, PIIMiddleware,   ModelCallLimitMiddleware,ModelFallbackMiddleware,LLMToolSelectorMiddleware,ContextEditingMiddleware,ClearToolUsesEdit,FilesystemFileSearchMiddleware
from langchain.agents.structured_output import ProviderStrategy, ToolStrategy, AutoStrategy,    MultipleStructuredOutputsError, StructuredOutputValidationError
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
from langchain.agents import create_agent
import os
from pydantic import BaseModel, Field
from rich import print as rprint
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage, ToolMessage
from langchain.messages import RemoveMessage
from langgraph.graph.message import REMOVE_ALL_MESSAGES
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import before_agent,before_model,after_agent,after_model
from typing import  Any,Callable
from langgraph.runtime import Runtime
from langchain.agents import AgentState
from langchain.agents.middleware import ModelCallResult,AgentMiddleware,wrap_model_call,ModelResponse,ModelRequest,wrap_tool_call,ToolCallRequest
from langgraph.checkpoint.postgres import PostgresSaver
load_dotenv(override=True)
my_api_key = os.getenv("API_KEY")
my_base_url = os.getenv("BASE_URL")
#1 初始化大模型
myModel = init_chat_model(
    model="qwen3.8-max",
    model_provider="openai",
    api_key=my_api_key,
    base_url=my_base_url,
    timeout=300,
    max_retries=3,
    # reasoning_effort="max"#思考模式    #'none', 'minimal', 'low', 'medium', 'high', 'xhigh', 'max'",
)

mycheckpoint=InMemorySaver()
#4初始化1个agent实例
myAgent=create_agent(
    model=myModel,
    name="琳琳",
    checkpointer=mycheckpoint,
    middleware=[
        SummarizationMiddleware(
            model=myModel,
            trigger=[  ("tokens",50),  ("messages",4),  ],
            keep=("messages",4),
            summary_prompt=(
                "你是一个对话摘要助手。请将以下历史对话压缩为一段简洁的中文摘要。"
                "要求："
                "1. 仅输出摘要正文，不要包含英文，尽量用一句话"
                "2. 保留关键实体（人名、地名、时间、意图）。"
                "3. 字数控制在100字以内。"

             )
        )
    ]
)
#4 创建1个config文件，保证同1个thread id是共享记忆的
myConfig={
    "configurable":{
        "thread_id":"1"
    }
}

myAgent.invoke({"messages": [SystemMessage("你是1个两性情感的AI助手，拥有几十年的两性问题经验，回答问题的语气自然，输出答案前会仔细检查内容真实性")]},myConfig)
myAgent.invoke({"messages": [HumanMessage("你好我是小向")]},myConfig)
myAgent.invoke({"messages": [ HumanMessage("跟高兴认识你")]},myConfig)
myAgent.invoke({"messages": [ HumanMessage("我是男生，身高175，体重82kg，最近在健身，然后失业了，在学习langchin框架，fastapi，pyPDF,准备找ai大模型应用相关的岗位")]},myConfig)
myAgent.invoke({"messages": [ HumanMessage("我是1996年出生的，本科学历，北京语言大学计算机专业的，然后是19年毕业，在中国兵器集团下面的昆明远达光学有限公司工作过2年，然后回重庆做了2年的软件实施，最近2年在重庆的赛力斯集团工作，负责运维和资产管理，也处理一些领导的紧急需求，但是都不涉及开发，上一分工作的年薪大概13w")]},myConfig)
final_response=myAgent.invoke({"messages": [ HumanMessage("请你给我一些职业选择上的建议，我想工资多点，但是不想离开重庆？")]},myConfig)
for msg in final_response["messages"]:
        msg.pretty_print()

================================ Human Message =================================

Here is a summary of the conversation to date:

用户要求将历史对话压缩为百字内中文一句话摘要，并保留人名、地名、时间等关键实体。
================================== Ai Message ==================================
Name: 琳琳

很高兴认识你，兄弟。听起来你现在处在一个“人生重启”的阶段：一边健身调整身体，一边学技术准备转/回到 AI 应用方向。这个状态其实挺好的，虽然失业会有压力，但你在主动做事，没有停摆。

我先帮你梳理一下你现在手里的牌：

### 你的优势
1. **在健身**
   - 身高 175，体重 82kg，如果体脂不算太高，这个身材基础不差。
   - 健身对失业期特别重要，它能稳定情绪、提升精力，让你面试状态更好。

2. **正在学 AI 应用开发**
   - LangChain / FastAPI / pyPDF 这条线是比较实用的。
   - 现在不少 AI 应用岗位确实需要：
     - LLM 应用开发
     - RAG 检索增强
     - 文档解析
     - API 服务
     - Agent 工作流
     - Prompt 工程
     - 向量数据库
     - 后端接口能力

3. **目标相对明确**
   - 你不是完全迷茫，而是知道要往“AI 大模型应用相关岗位”走，这已经比很多人强了。

---

## 你现在最该做的，不是焦虑，而是把学习变成“能求职的作品”

很多失业期学技术的人容易陷入一个坑：  
一直学，但拿不出一个像样的项目。

你要尽快从“我学了什么”转成“我做过什么”。

比如你可以做一个这样的项目：

### 项目方向：企业知识库问答系统
这个岗位很常见，也很适合你现在的技术栈。

功能可以是：

- 上传 PDF 文档
- 解析 PDF 内容
- 文档切分
- 存入向量数据库
- 用户提问
- 检索相关内容
- 调用大模型生成回答
- 支持引用来源
- 提供 FastAPI 接口
- 